In [1]:
here::i_am("snakemake/01_create_arrow.R")
source(here::here("settings.R"))

# I/O
io$output.directory <- file.path(io$basedir,"ArchR_test3")
dir.create(file.path(io$output.directory), showWarnings = FALSE)

setwd(io$output.directory)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_final

Setting default number of Parallel threads to 1.



In [2]:
args = list()
args$sample = 'BGRGP1'
args$min_fragments = 10000
args$min_tss_score = 2

In [3]:
#genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)
genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[1:200]
geneAnnotation$genes = geneAnnotation$genes[as.vector(seqnames(geneAnnotation$genes)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$exons = geneAnnotation$exons[as.vector(seqnames(geneAnnotation$exons)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$TSS = geneAnnotation$TSS[as.vector(seqnames(geneAnnotation$TSS)) %in% genomeAnnotation$chromSizes@seqnames@values]

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [4]:
geneAnnotation$genes

GRanges object with 25970 ranges and 2 metadata columns:
           seqnames        ranges strand |            gene_id
              <Rle>     <IRanges>  <Rle> |        <character>
      [1]      chr1   15188-30379      + | ENSOCUG00000014251
      [2]      chr1   52453-53038      + | ENSOCUG00000005054
      [3]      chr1   57421-74906      - | ENSOCUG00000005046
      [4]      chr1   75200-85749      + | ENSOCUG00000005044
      [5]      chr1   92423-95846      - | ENSOCUG00000005040
      ...       ...           ...    ... .                ...
  [25966] chrUn0178 268869-270549      + | ENSOCUG00000033412
  [25967] chrUn0178 312451-325326      - | ENSOCUG00000000865
  [25968] chrUn0178 340582-341913      - | ENSOCUG00000028016
  [25969] chrUn0178 421791-423600      + | ENSOCUG00000031685
  [25970] chrUn0178 476220-476559      - | ENSOCUG00000021865
                      symbol
                 <character>
      [1]              WDR31
      [2]             RNF183
      [3]            

In [5]:
# trim 2kb ends of geneAnnotation, otherwise gives error: 

exclude = GRanges(
    seqnames = Rle(rep(genomeAnnotation$chromSizes@seqnames@values,2)),
    ranges = IRanges(start = c(genomeAnnotation$chromSizes@ranges@start, 
                               genomeAnnotation$chromSizes@ranges@width-2000), 
                     end = c(genomeAnnotation$chromSizes@ranges@start + 2000, 
                             genomeAnnotation$chromSizes@ranges@width)))

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$TSS))
if(nrow(exclude_ranges)){
geneAnnotation$TSS = geneAnnotation$TSS[-exclude_ranges$subjectHits]
}

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$genes))
if(nrow(exclude_ranges)){
geneAnnotation$genes = geneAnnotation$genes[-exclude_ranges$subjectHits]
}
exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$exons))
if(nrow(exclude_ranges)){
geneAnnotation$exons = geneAnnotation$exons[-exclude_ranges$subjectHits]
}

In [6]:
geneAnnotation$TSS
geneAnnotation$genes
geneAnnotation$exons
genomeAnnotation$chromSizes 

GRanges object with 46272 ranges and 2 metadata columns:
           seqnames    ranges strand |     tx_id            tx_name
              <Rle> <IRanges>  <Rle> | <integer>        <character>
      [1]      chr1     15188      + |         1 ENSOCUT00000014253
      [2]      chr1     20324      + |         2 ENSOCUT00000038745
      [3]      chr1     52453      + |         3 ENSOCUT00000005053
      [4]      chr1     75200      + |         4 ENSOCUT00000005041
      [5]      chr1    125890      + |         5 ENSOCUT00000005032
      ...       ...       ...    ... .       ...                ...
  [46268] chrUn0178    325325      - |     47098 ENSOCUT00000034796
  [46269] chrUn0178    325307      - |     47099 ENSOCUT00000000866
  [46270] chrUn0178    324997      - |     47100 ENSOCUT00000063424
  [46271] chrUn0178    341912      - |     47101 ENSOCUT00000024657
  [46272] chrUn0178    476558      - |     47102 ENSOCUT00000014523
  -------
  seqinfo: 1402 sequences from an unspecified gen

GRanges object with 25912 ranges and 2 metadata columns:
           seqnames        ranges strand |            gene_id
              <Rle>     <IRanges>  <Rle> |        <character>
      [1]      chr1   15188-30379      + | ENSOCUG00000014251
      [2]      chr1   52453-53038      + | ENSOCUG00000005054
      [3]      chr1   57421-74906      - | ENSOCUG00000005046
      [4]      chr1   75200-85749      + | ENSOCUG00000005044
      [5]      chr1   92423-95846      - | ENSOCUG00000005040
      ...       ...           ...    ... .                ...
  [25908] chrUn0178 268869-270549      + | ENSOCUG00000033412
  [25909] chrUn0178 312451-325326      - | ENSOCUG00000000865
  [25910] chrUn0178 340582-341913      - | ENSOCUG00000028016
  [25911] chrUn0178 421791-423600      + | ENSOCUG00000031685
  [25912] chrUn0178 476220-476559      - | ENSOCUG00000021865
                      symbol
                 <character>
      [1]              WDR31
      [2]             RNF183
      [3]            

GRanges object with 213647 ranges and 3 metadata columns:
            seqnames        ranges strand |   exon_id            gene_id
               <Rle>     <IRanges>  <Rle> | <integer>        <character>
       [1]      chr1   15188-15282      + |         1 ENSOCUG00000014251
       [2]      chr1   20324-20326      + |         2 ENSOCUG00000014251
       [3]      chr1   20453-20603      + |         3 ENSOCUG00000014251
       [4]      chr1   20453-20603      + |         4 ENSOCUG00000014251
       [5]      chr1   21033-21163      + |         5 ENSOCUG00000014251
       ...       ...           ...    ... .       ...                ...
  [213643] chrUn0178 340582-340639      - |    215864 ENSOCUG00000028016
  [213644] chrUn0178 341454-341747      - |    215865 ENSOCUG00000028016
  [213645] chrUn0178 341783-341913      - |    215866 ENSOCUG00000028016
  [213646] chrUn0178 476220-476410      - |    215867 ENSOCUG00000021865
  [213647] chrUn0178 476521-476559      - |    215868 ENSOCUG00000

GRanges object with 200 ranges and 0 metadata columns:
         seqnames      ranges strand
            <Rle>   <IRanges>  <Rle>
    [1]      chr1 1-194850757      *
    [2]     chr10  1-47997241      *
    [3]     chr11  1-87554214      *
    [4]     chr12 1-155355395      *
    [5]     chr13 1-143360832      *
    ...       ...         ...    ...
  [196] chrUn0174    1-700076      *
  [197] chrUn0175    1-571908      *
  [198] chrUn0176    1-557607      *
  [199] chrUn0177    1-597847      *
  [200] chrUn0178    1-577984      *
  -------
  seqinfo: 3242 sequences from an unspecified genome

In [7]:
fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation, 
    force= TRUE
)

ArchR logging to : ArchRLogs/ArchR-createArrows-4a474a946992-Date-2022-01-28_Time-12-57-04.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 12:57:04 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 12:57:04 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0.001 mins elapsed.

2022-01-28 12:57:04 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:00:37 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 1 Percent, 3.55 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:03:25 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile

2022-01-28 13:31:14 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 31 Percent, 34.17 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:31:25 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 32 Percent, 34.35 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:31:35 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 33 Percent, 34.517 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:31:46 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 34 Percent, 34.687 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'

2022-01-28 13:38:03 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 73 Percent, 40.986 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:38:13 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 74 Percent, 41.151 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:38:22 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 75 Percent, 41.302 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 13:38:32 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 76 Percent, 41.46 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent


************************************************************
2022-01-28 14:25:47 : ERROR Found in .fastFeatureCounts for (rabbit_BGRGP1 : 1 of 1) 
LogFile = ArchRLogs/ArchR-createArrows-4a474a946992-Date-2022-01-28_Time-12-57-04.log

<simpleError in .getFragsFromArrow(ArrowFile = ArrowFile, chr = names(featureList)[x],     out = "IRanges", cellNames = cellNames): Chromosome NA not in ArrowFile! Available Chromosomes are : chr1,chr10,chr11,chr12,chr13,chr14,chr15,chr16,chr17,chr18,chr19,chr2,chr20,chr21,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chrUn0001,chrUn0002,chrUn0003,chrUn0004,chrUn0005,chrUn0006,chrUn0007,chrUn0008,chrUn0009,chrUn0010,chrUn0011,chrUn0012,chrUn0013,chrUn0014,chrUn0015,chrUn0016,chrUn0017,chrUn0018,chrUn0019,chrUn0020,chrUn0021,chrUn0022,chrUn0023,chrUn0024,chrUn0025,chrUn0026,chrUn0027,chrUn0028,chrUn0029,chrUn0030,chrUn0031,chrUn0032,chrUn0033,chrUn0034,chrUn0035,chrUn0036,chrUn0037,chrUn0038,chrUn0039,chrUn0040,chrUn0041,chrUn0042,chrUn0043,chrUn0044,chrUn0045,chrUn0

createArrowFiles has encountered an error, checking if any ArrowFiles completed..

2022-01-28 14:25:47 : 

ArchR logging successful to : ArchRLogs/ArchR-createArrows-4a474a946992-Date-2022-01-28_Time-12-57-04.log



# Excude chrs with 0 fragments?

In [16]:
#genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)
exclude = c('chrUn0141', 'chrUn0162', 'chrUn0165')
genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[1:200]
genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[!genomeAnnotation$chromSizes@seqnames@values %in% exclude]

geneAnnotation$genes = geneAnnotation$genes[as.vector(seqnames(geneAnnotation$genes)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$exons = geneAnnotation$exons[as.vector(seqnames(geneAnnotation$exons)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$TSS = geneAnnotation$TSS[as.vector(seqnames(geneAnnotation$TSS)) %in% genomeAnnotation$chromSizes@seqnames@values]

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [17]:
# trim 2kb ends of geneAnnotation, otherwise gives error: 

exclude = GRanges(
    seqnames = Rle(rep(genomeAnnotation$chromSizes@seqnames@values,2)),
    ranges = IRanges(start = c(genomeAnnotation$chromSizes@ranges@start, 
                               genomeAnnotation$chromSizes@ranges@width-2000), 
                     end = c(genomeAnnotation$chromSizes@ranges@start + 2000, 
                             genomeAnnotation$chromSizes@ranges@width)))

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$TSS))
if(nrow(exclude_ranges)){
geneAnnotation$TSS = geneAnnotation$TSS[-exclude_ranges$subjectHits]
}

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$genes))
if(nrow(exclude_ranges)){
geneAnnotation$genes = geneAnnotation$genes[-exclude_ranges$subjectHits]
}
exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$exons))
if(nrow(exclude_ranges)){
geneAnnotation$exons = geneAnnotation$exons[-exclude_ranges$subjectHits]
}

In [18]:
geneAnnotation$TSS
geneAnnotation$genes
geneAnnotation$exons
genomeAnnotation$chromSizes 

GRanges object with 46265 ranges and 2 metadata columns:
           seqnames    ranges strand |     tx_id            tx_name
              <Rle> <IRanges>  <Rle> | <integer>        <character>
      [1]      chr1     15188      + |         1 ENSOCUT00000014253
      [2]      chr1     20324      + |         2 ENSOCUT00000038745
      [3]      chr1     52453      + |         3 ENSOCUT00000005053
      [4]      chr1     75200      + |         4 ENSOCUT00000005041
      [5]      chr1    125890      + |         5 ENSOCUT00000005032
      ...       ...       ...    ... .       ...                ...
  [46261] chrUn0178    325325      - |     47098 ENSOCUT00000034796
  [46262] chrUn0178    325307      - |     47099 ENSOCUT00000000866
  [46263] chrUn0178    324997      - |     47100 ENSOCUT00000063424
  [46264] chrUn0178    341912      - |     47101 ENSOCUT00000024657
  [46265] chrUn0178    476558      - |     47102 ENSOCUT00000014523
  -------
  seqinfo: 1402 sequences from an unspecified gen

GRanges object with 25907 ranges and 2 metadata columns:
           seqnames        ranges strand |            gene_id
              <Rle>     <IRanges>  <Rle> |        <character>
      [1]      chr1   15188-30379      + | ENSOCUG00000014251
      [2]      chr1   52453-53038      + | ENSOCUG00000005054
      [3]      chr1   57421-74906      - | ENSOCUG00000005046
      [4]      chr1   75200-85749      + | ENSOCUG00000005044
      [5]      chr1   92423-95846      - | ENSOCUG00000005040
      ...       ...           ...    ... .                ...
  [25903] chrUn0178 268869-270549      + | ENSOCUG00000033412
  [25904] chrUn0178 312451-325326      - | ENSOCUG00000000865
  [25905] chrUn0178 340582-341913      - | ENSOCUG00000028016
  [25906] chrUn0178 421791-423600      + | ENSOCUG00000031685
  [25907] chrUn0178 476220-476559      - | ENSOCUG00000021865
                      symbol
                 <character>
      [1]              WDR31
      [2]             RNF183
      [3]            

GRanges object with 213632 ranges and 3 metadata columns:
            seqnames        ranges strand |   exon_id            gene_id
               <Rle>     <IRanges>  <Rle> | <integer>        <character>
       [1]      chr1   15188-15282      + |         1 ENSOCUG00000014251
       [2]      chr1   20324-20326      + |         2 ENSOCUG00000014251
       [3]      chr1   20453-20603      + |         3 ENSOCUG00000014251
       [4]      chr1   20453-20603      + |         4 ENSOCUG00000014251
       [5]      chr1   21033-21163      + |         5 ENSOCUG00000014251
       ...       ...           ...    ... .       ...                ...
  [213628] chrUn0178 340582-340639      - |    215864 ENSOCUG00000028016
  [213629] chrUn0178 341454-341747      - |    215865 ENSOCUG00000028016
  [213630] chrUn0178 341783-341913      - |    215866 ENSOCUG00000028016
  [213631] chrUn0178 476220-476410      - |    215867 ENSOCUG00000021865
  [213632] chrUn0178 476521-476559      - |    215868 ENSOCUG00000

GRanges object with 197 ranges and 0 metadata columns:
         seqnames      ranges strand
            <Rle>   <IRanges>  <Rle>
    [1]      chr1 1-194850757      *
    [2]     chr10  1-47997241      *
    [3]     chr11  1-87554214      *
    [4]     chr12 1-155355395      *
    [5]     chr13 1-143360832      *
    ...       ...         ...    ...
  [193] chrUn0174    1-700076      *
  [194] chrUn0175    1-571908      *
  [195] chrUn0176    1-557607      *
  [196] chrUn0177    1-597847      *
  [197] chrUn0178    1-577984      *
  -------
  seqinfo: 3242 sequences from an unspecified genome

In [19]:
fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation, 
    force= TRUE
)

ArchR logging to : ArchRLogs/ArchR-createArrows-4a47264f631e-Date-2022-01-28_Time-14-30-36.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 14:30:37 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 14:30:37 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0 mins elapsed.

2022-01-28 14:30:37 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 14:33:44 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 1 Percent, 3.106 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 14:36:23 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 2 Perc

2022-01-28 15:03:02 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 31 Percent, 32.415 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:03:13 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 32 Percent, 32.592 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:03:23 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 34 Percent, 32.759 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:03:33 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 35 Percent, 32.928 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percen

2022-01-28 15:08:29 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 65 Percent, 37.86 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:08:38 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 66 Percent, 38.013 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:08:47 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 67 Percent, 38.161 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:08:56 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 68 Percent, 38.315 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent

2022-01-28 15:13:32 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 98 Percent, 42.915 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:13:41 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 99 Percent, 43.064 mins elapsed.

2022-01-28 15:13:46 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Temporary File, 43.15 mins elapsed.

2022-01-28 15:13:46 : (rabbit_BGRGP1 : 1 of 1) Creating ArrowFile From Temporary File, 43.15 mins elapsed.

2022-01-28 15:36:51 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Arrow File, 66.223 mins elapsed.

2022-01-28 15:48:09 : (rabbit_BGRGP1 : 1 of 1) CellStats : Number of Cells Pass Filter = 11220 , 77.527 mins elapsed.

2022-01-28 15:48:09 : (rabbit_BGRGP1 : 1 of 1) CellStats : Median Frags = 30336.5 , 77.528 mins elapsed.

2022-01-28 15:48:09 : (rabbit_BGRGP1 : 1 of 1) CellStats : Median TSS Enrichment = 3.551 , 77.528 mins ela

In [21]:
ArrowFiles

[1] "rabbit_BGRGP1.arrow"

In [20]:
# Calculate doublet scores

ArrowFile = paste0(io$output.directory, '/rabbit_', args$sample, '.arrow')

doubScores <- addDoubletScores(
  input = ArrowFiles,
  k = 15, #Refers to how many cells near a "pseudo-doublet" to count.
  knnMethod = "UMAP", #Refers to the embedding to use for nearest neighbor search.
  LSIMethod = 1
)


ArchR logging to : ArchRLogs/ArchR-addDoubletScores-4a4734e76002-Date-2022-01-28_Time-16-50-39.log
If there is an issue, please report to github with logFile!

2022-01-28 16:50:39 : Batch Execution w/ safelapply!, 0 mins elapsed.

2022-01-28 16:50:39 : rabbit_BGRGP1 (1 of 1) :  Computing Doublet Statistics, 0 mins elapsed.

Warning message:
“package ‘Seurat’ was built under R version 4.1.2”



************************************************************
2022-01-28 17:02:12 : ERROR Found in uwot::umap for rabbit_BGRGP1 (1 of 1) :  
LogFile = ArchRLogs/ArchR-addDoubletScores-4a4734e76002-Date-2022-01-28_Time-16-50-39.log

<simpleError in annoy_search_parallel(X = X, k = k, ann = ann, search_k = search_k,     tmpdir = tmpdir, n_threads = n_threads, grain_size = grain_size,     verbose = verbose): search_k/n_trees settings were unable to find 40 neighbors for all items.>

************************************************************



ERROR: Error in .logError(e, fn = "uwot::umap", info = prefix, errorList = errorList, : Exiting See Error Above
